# 📖 Bible Reading Progress Tracker — Pipeline Evaluation

**Purpose**  
Evaluate the NER pipeline on a representative sample before running on the full 16k dataset.

**Sampling strategy**  
Select 7 days where the dominant book changes — ensures coverage of different book names,
abbreviation styles, and chapter ranges without redundant repetition of the same pattern.

| # | Section | Description |
|---|---------|-------------|
| 0 | **Setup** | Imports, paths, orchestrator init |
| 1 | **Sample Selection** | Find 7 days where dominant book changes |
| 2 | **Extract** | Run NER extraction + validate output |
| 3 | **Persist** | Write to DB + sanity check row counts |
| 4 | **Schedule** | Validate reading plan for sampled dates |
| 5 | **Summarize** | Print daily compliance per sampled date |

## 0. Setup

In [1]:
import sys
import warnings
import pandas as pd
from pathlib import Path
from datetime import date
from typing import Tuple
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
root = Path('..').resolve() # project root
sys.path.append(str(root / 'src'))

from bible_data import load_bible_data
from config.settings import PROCESSED_DIR, MODEL_PATH, READING_PLAN_PATH
from preprocessing.normalization import BibleReferenceNormalizer
from sessions.database import get_session, create_tables
from sessions.database import Member, Message, BibleReference, ReadingProgress
from pipelines import BibleProgressPipeline


# ── Paths ─────────────────────────────────────────────────────────────────────
OUT_DIR = PROCESSED_DIR / 'pipeline_evaluation'
OUT_DIR.mkdir(parents=True, exist_ok=True)

bible_data = load_bible_data()

print('Setup complete.')

Setup complete.


In [2]:
bible_pipeline = BibleProgressPipeline(
    saved_path=MODEL_PATH,
    bible_books=bible_data['books'],
    plan_path=READING_PLAN_PATH
)
print('Pipeline ready.')
print(f'  Reading Plan : {bible_pipeline.schedule.total_days} days ({bible_pipeline.schedule.dates[0]} → {bible_pipeline.schedule.dates[-1]})')

2026-04-20 07:32:37 | INFO     | bible_pipeline.extraction.extractor | Loading model from saved path: C:\one one\Desktop\bible_reading_recap_nlp\models\indobert-bible-ner-v4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Pipeline ready.
  Reading Plan : 7 days (2020-08-03 → 2020-08-09)


### Helper Functions

In [3]:
def select_sample(
        df: pd.DataFrame,
        start_date: date,
        end_date: date, 
) -> pd.DataFrame:
    """Filter messages by date range."""
    sample_df = df[
        (df['date'] >= start_date) &
        (df['date'] <= end_date)
    ].copy().reset_index(drop=True)
    sample_df.index.name = 'message_id'
    sample_df = sample_df.reset_index()
    return sample_df

def run_extraction(
        sample_df: pd.DataFrame,
        pipeline: 'BibleProgressPipeline',
) -> pd.DataFrame:
    """Run NER extraction over sample dataframe using the given pipeline"""
    tqdm.pandas()
    sample_df['ner_references'] = sample_df['message'].progress_apply(
    lambda msg: pipeline.extractor.extract(msg)
    )
    sample_df['ner_ref_count'] = sample_df['ner_references'].apply(len)
    return sample_df

def check_zero_refs(sample_df: pd.DataFrame) -> pd.DataFrame:
    """Return and display messages with zero extracted references."""
    zero_refs = sample_df[sample_df['ner_ref_count'] == 0]
    return zero_refs

def run_normalization(
        sample_df: pd.DataFrame,
        normalizer: BibleReferenceNormalizer,
) -> Tuple[pd.DataFrame, 'BibleReferenceNormalizer', pd.DataFrame]:
    """Normalize references and identify invalid ones."""
    sample_df = sample_df.copy()
    sample_df['ner_references'] = sample_df['ner_references'].apply(normalizer.normalize)

    invalid_rows = sample_df[
        sample_df['ner_references'].apply(
            lambda refs: any(not r['is_valid'] for r in refs)
        )
    ]
    return sample_df, normalizer, invalid_rows

def print_summary(
        sample_df: pd.DataFrame,
        normalizer: 'BibleReferenceNormalizer',
        invalid_rows: pd.DataFrame,
) -> None:
    """Print extraction summary table and resolve stats."""
    total = len(sample_df)
    has_refs = (sample_df['ner_ref_count'] > 0).sum()
    no_refs = (sample_df['ner_ref_count'] == 0).sum()
    n_invalid = len(invalid_rows)


    summary = pd.DataFrame([
        ('Total sampled', total, ''),
        ('References extracted', has_refs, f'{has_refs/total*100:.1f}%'),
        ('Zero references', no_refs, f'{no_refs/total*100:.1f}%'),
        ('Invalid references', n_invalid, f'{n_invalid/total*100:.1f}%'),
    ], columns=['Metric', 'Count', 'Pct'])

    print(summary.to_string(index=False))

    print('\nResolver stats:')
    for method, count in normalizer.get_stats().items():
        print(f'  {str(method):25s}: {count}')

def explode_refs(sample_df: pd.DataFrame) -> pd.DataFrame:
    """Explode ner_references column into one row per individual reference."""
    refs_df = (
        sample_df[sample_df['ner_ref_count'] > 0]
        [['message_id', 'sender', 'message', 'ner_references']]
        .explode('ner_references')
        .reset_index(drop=True)
    )

    refs_df = pd.concat([
        refs_df[['message_id', 'sender', 'message']],
        refs_df['ner_references'].apply(pd.Series)
    ], axis=1)
    
    return refs_df

def save_outputs(
        sample_df: pd.DataFrame,
        refs_df: pd.DataFrame,
        zero_refs: pd.DataFrame,
        out_dir: Path,
        label: str = '',
) -> None:
    """Save pipeline outputs (sample, references, zero-refs) to CSV format."""
    suffix = f'_{label}' if label else ''
    sample_df.to_csv(out_dir / f'pipeline_sample{suffix}.csv', index=False, encoding='utf-8-sig')
    refs_df.to_csv(out_dir / f'extracted_references{suffix}.csv', index=False, encoding='utf-8-sig')
    zero_refs[['message_id', 'sender', 'message']].to_csv(
        out_dir / f'zero_refs{suffix}.csv', index=False, encoding='utf-8-sig'
    )

    print(f'Saved pipeline_sample{suffix}.csv      → {len(sample_df):,} rows')
    print(f'Saved extracted_references{suffix}.csv → {len(refs_df):,} rows')
    print(f'Saved zero_refs{suffix}.csv            → {len(zero_refs):,} rows')

Run NER extraction over the sampled messages using `BibleReferenceExtractor`.
Validation checks:
- **Zero refs** — progress messages that returned nothing
- **Invalid refs** — references where normalization flagged `is_valid=False`
- **Summary table** — overall extraction stats + resolver method breakdown
- **Spot-check** — verbose single-message debug

---
## 1. Sample Selection

Find 7 days where the dominant book changes — spread across the full date range.  
**Dominant book** = the book that appears most in progress messages on that day.

In [4]:
df = pd.read_csv(PROCESSED_DIR / 'reviewed_messages.csv')

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['date'] = df['timestamp'].dt.date
df['is_progress'] = df['category'].str.strip().str.lower() == 'report'

progress_df = df[df['is_progress']].copy().reset_index(drop=True)

print(f'Total parsed messages: {len(progress_df):,}')
print(f'Date range: {progress_df["date"].min()} → {progress_df["date"].max()}')

Total parsed messages: 16,539
Date range: 2020-08-03 → 2022-06-25


## 2. Samples

Each sample covers a 7-day window. Run all cells in a sample block to get the full evaluation for that window. The last block that was run sets `sample_df`.

### Sample 1 — 2020-08-03 to 2020-08-09

In [5]:
sample1_df = select_sample(progress_df, start_date=date(2020, 8, 3), end_date=date(2020, 8, 9))

print(f'Sampled messages : {len(sample1_df)}')
print(f'Across {sample1_df['date'].nunique()} days')
print(f'By {sample1_df['sender'].nunique()} unique senders')
print()
sample1_df.groupby('date')['message'].count().rename('message_count')

Sampled messages : 257
Across 7 days
By 47 unique senders



date
2020-08-03    40
2020-08-04    46
2020-08-05    42
2020-08-06    40
2020-08-07    42
2020-08-08    40
2020-08-09     7
Name: message_count, dtype: int64

In [6]:
sample1_df = run_extraction(sample1_df, bible_pipeline)

print(f'Done. {sample1_df["ner_ref_count"].sum():,} total references extracted.')

  0%|          | 0/257 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Done. 272 total references extracted.


In [7]:
zero_refs_s1 = check_zero_refs(sample1_df)

print(f'Messages with zero refs : {len(zero_refs_s1):,} ({len(zero_refs_s1) / len(sample1_df) * 100:.1f}%)')
print()
zero_refs_s1[['date', 'sender', 'message']].head(20)

Messages with zero refs : 0 (0.0%)



,date,sender,message


In [8]:
normalizer_s1 = BibleReferenceNormalizer()

# 2.2 Normalization + Invalid References
sample1_df, normalizer_s1, invalid_rows_s1 = run_normalization(sample1_df, normalizer_s1)

print(f'Messages with invalid refs : {len(invalid_rows_s1):,}')
for _, row in invalid_rows_s1.head(10).iterrows():
    invalid = [r for r in row['ner_references'] if not r['is_valid']]
    print(f"  {row['message']!r:40s} → {invalid}")

Messages with invalid refs : 0


In [9]:
# 2.3 Extraction Summary
print_summary(sample1_df, normalizer_s1, invalid_rows_s1)

              Metric  Count    Pct
       Total sampled    257       
References extracted    257 100.0%
     Zero references      0   0.0%
  Invalid references      0   0.0%

Resolver stats:
  exact                    : 272
  fuzzy                    : 0
  failed                   : 0


In [10]:
# 2.4 Exploded Results
refs_df_s1 = explode_refs(sample1_df)

print(f'Shape : {refs_df_s1.shape}')
refs_df_s1.head(10)

Shape : (272, 8)


,message_id,sender,message,book_start,start_chapter,book_end,end_chapter,is_valid
0,0,Melanie Chandra,Kej 1-2 done,Kejadian,1,Kejadian,2,True
1,1,Lindawati Haryanto,Kej 1-2 done,Kejadian,1,Kejadian,2,True
2,2,Sherly Cahyadi,Kej 1-2 done,Kejadian,1,Kejadian,2,True
3,3,Seto Ninik,Kej 1-2 done,Kejadian,1,Kejadian,2,True
4,4,🪸Martha 🍁,Kej 1-2 done,Kejadian,1,Kejadian,2,True
5,5,Dewi Pratiwi,Kej 1-2 done,Kejadian,1,Kejadian,2,True
6,6,Endang Surati,Kej 1- 2 selesai.🙏,Kejadian,1,Kejadian,2,True
7,7,Dicky Andrian,Kej 1-2 done,Kejadian,1,Kejadian,2,True
8,8,🎍,Kej 1-2 done,Kejadian,1,Kejadian,2,True
9,9,"dr. Andreas C.N., Sp.B.",Kej 1-2 selesai,Kejadian,1,Kejadian,2,True


In [ ]:
save_outputs(sample1_df, refs_df_s1, zero_refs_s1, OUT_DIR, label='s1')

Saved pipeline_sample_s1.csv      → 257 rows
Saved extracted_references_s1.csv → 272 rows
Saved zero_refs_s1.csv            → 0 rows


### Sample 2 — 2021-12-06 to 2021-12-12

In [11]:
sample2_df = select_sample(progress_df, start_date=date(2021, 12, 6), end_date=date(2021, 12, 12))

print(f'Sampled messages : {len(sample2_df)}')
print(f'Across {sample2_df['date'].nunique()} days')
print(f'By {sample2_df['sender'].nunique()} unique senders')
print()
sample2_df.groupby('date')['message'].count().rename('message_count')

Sampled messages : 137
Across 7 days
By 36 unique senders



date
2021-12-06    26
2021-12-07    22
2021-12-08    24
2021-12-09    15
2021-12-10    19
2021-12-11    26
2021-12-12     5
Name: message_count, dtype: int64

In [12]:
sample2_df = run_extraction(sample2_df, bible_pipeline)

print(f'Done. {sample2_df["ner_ref_count"].sum():,} total references extracted.')

  0%|          | 0/137 [00:00<?, ?it/s]

2026-04-20 07:35:29 | WARNING  | bible_pipeline.extraction.ner_parser | No book token and no context book at pos 0; skipping token ('NUM', '2')
2026-04-20 07:35:29 | WARNING  | bible_pipeline.extraction.ner_parser | No book token and no context book at pos 1; skipping token ('RANGE', '-')
2026-04-20 07:35:29 | WARNING  | bible_pipeline.extraction.ner_parser | No book token and no context book at pos 2; skipping token ('NUM', '3')
2026-04-20 07:35:29 | WARNING  | bible_pipeline.extraction.extractor | NER span produced no parsed ref: '2 - 3'
Done. 162 total references extracted.


In [13]:
zero_refs_s2 = check_zero_refs(sample2_df)

print(f'Messages with zero refs : {len(zero_refs_s2):,} ({len(zero_refs_s2) / len(sample2_df) * 100:.1f}%)')
print()
zero_refs_s2[['date', 'sender', 'message']].head(20)

Messages with zero refs : 2 (1.5%)



,date,sender,message
67,2021-12-08,Ibu Kartika,_Dan 2-3_ ✓
122,2021-12-11,Wesley and Mom😇🥰,Yehezkiel✔️


In [14]:
normalizer_s2 = BibleReferenceNormalizer()

sample2_df, normalizer_s2, invalid_rows_s2 = run_normalization(sample2_df, normalizer_s2)

print(f'Messages with invalid refs : {len(invalid_rows_s2):,}')
for _, row in invalid_rows_s2.head(10).iterrows():
    invalid = [r for r in row['ner_references'] if not r['is_valid']]
    print(f"  {row['message']!r:40s} → {invalid}")

Messages with invalid refs : 0


In [15]:
print_summary(sample2_df, normalizer_s2, invalid_rows_s2)

              Metric  Count   Pct
       Total sampled    137      
References extracted    135 98.5%
     Zero references      2  1.5%
  Invalid references      0  0.0%

Resolver stats:
  exact                    : 181
  fuzzy                    : 0
  failed                   : 0


In [16]:
refs_df_s2 = explode_refs(sample2_df)

print(f'Shape : {refs_df_s2.shape}')
refs_df_s2.head(10)

Shape : (162, 8)


,message_id,sender,message,book_start,start_chapter,book_end,end_chapter,is_valid
0,0,Lindawati Haryanto,Yeh 46-47 done,Yehezkiel,46,Yehezkiel,47,True
1,1,Sim Ay Tjan,Yeh 46-47 done,Yehezkiel,46,Yehezkiel,47,True
2,2,Ci Ina Paperku,Yeh 40-41 done,Yehezkiel,40,Yehezkiel,41,True
3,3,Seto Ninik,Yeh 46-47 done,Yehezkiel,46,Yehezkiel,47,True
4,4,Melanie Chandra,Yehez finish,Yehezkiel,1,Yehezkiel,48,True
5,5,Yozef Tjandra,Yeh 40-47 done,Yehezkiel,40,Yehezkiel,47,True
6,6,Villas/Hepihippo,Yeh 28-31 done,Yehezkiel,28,Yehezkiel,31,True
7,7,Ivan teguh,Yeh 46-47 done,Yehezkiel,46,Yehezkiel,47,True
8,8,Dewi Pratiwi,Yeh 44-45 done,Yehezkiel,44,Yehezkiel,45,True
9,9,Dewi Pratiwi,Yeh 46-47 done,Yehezkiel,46,Yehezkiel,47,True


In [ ]:
save_outputs(sample2_df, refs_df_s2, zero_refs_s2, OUT_DIR, label='s2')

Saved pipeline_sample_s2.csv      → 137 rows
Saved extracted_references_s2.csv → 162 rows
Saved zero_refs_s2.csv            → 2 rows


### Error Analysis

In [17]:
gt_s1 = pd.read_csv(OUT_DIR / 'ground_truth_s1.csv')
gt_s2 = pd.read_csv(OUT_DIR / 'ground_truth_s2.csv')
gt = pd.concat([gt_s1, gt_s2], ignore_index=True)

pred = pd.concat([refs_df_s1, refs_df_s2], ignore_index=True)

In [18]:
gt

,message_id,sender,message,book_start,start_chapter,book_end,end_chapter,is_valid
0,0,Melanie Chandra,Kej 1-2 done,Kejadian,1,Kejadian,2,True
1,1,Lindawati Haryanto,Kej 1-2 done,Kejadian,1,Kejadian,2,True
2,2,Sherly Cahyadi,Kej 1-2 done,Kejadian,1,Kejadian,2,True
3,3,Seto Ninik,Kej 1-2 done,Kejadian,1,Kejadian,2,True
4,4,🪸Martha 🍁,Kej 1-2 done,Kejadian,1,Kejadian,2,True
...,...,...,...,...,...,...,...,...
429,133,Nurcahaya Sihombing,Dan 8-9 done,Daniel,8,Daniel,9,True
430,134,🪸Martha 🍁,Dan 4 -9 done,Daniel,4,Daniel,9,True
431,135,Jeffry,Yesaya 65 - 66 done\r\nYeremia 1 - 2 done,Yesaya,65,Yesaya,66,True
432,135,Jeffry,Yesaya 65 - 66 done\r\nYeremia 1 - 2 done,Yeremia,1,Yeremia,2,True


In [20]:
def to_ref_set(df):
    return set(
        zip(df['message_id'], df['book_start'], df['start_chapter'], df['book_end'], df['end_chapter'])
    )

gt_set = to_ref_set(gt)
pred_set = to_ref_set(pred)

In [21]:
tp = gt_set & pred_set
fp = pred_set - gt_set
fn = gt_set - pred_set

precision = len(tp) / (len(tp) + len(fp)) if (tp or fp) else 0
recall = len(tp) / (len(tp) + len(fn)) if (tp or fn) else 0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

print(f'TP {len(tp):>4}  FP {len(fp):>4}  FN {len(fn):>4}')
print(f'Precision : {precision:.3f}')
print(f'Recall    : {recall:.3f}')
print(f'F1        : {f1_score:.3f}')

TP  420  FP    3  FN    3
Precision : 0.993
Recall    : 0.993
F1        : 0.993


In [23]:
fn_df = gt[gt.apply(
    lambda r: (r['message_id'], r['book_start'], r['start_chapter'], r['book_end'], r['end_chapter']) in fn, axis=1   
)]
print(fn_df[['message_id', 'message', 'book_start']].to_string())

     message_id               message book_start
326          46  _Yeh_ ✓\r\n_Dan 1_ ✓     Daniel
351          67           _Dan 2-3_ ✓     Daniel
413         122           Yehezkiel✔️  Yehezkiel


---
## 3. Persist

Write extracted references to DB via `process_batch`. Sanity check: row counts per table should reflect what was inserted.

In [11]:
create_tables()
print('Tables ready.')

Tables created at: C:\one one\Desktop\bible_reading_recap_nlp\data\bible_progress.db
Tables ready.


In [ ]:
totals = bible_pipeline.process_batch(sample1_df)

print(f'Inserted {totals["refs"]:,} references')
print(f'Inserted {totals["chapters"]:,} chapter progress rows')
print(f'Skipped  {totals["skipped"]:,} invalid references')

Persisting:   0%|          | 0/257 [00:00<?, ?it/s]

2026-04-12 18:36:52 | INFO     | bible_pipeline.pipelines | Batch complete — refs=272 chapters=587 skipped=0
Inserted 272 references
Inserted 587 chapter progress rows
Skipped  0 invalid references


### 3.1 DB Row Count Validation

In [13]:
with get_session() as session:
    n_members  = session.query(Member).count()
    n_messages = session.query(Message).count()
    n_refs     = session.query(BibleReference).count()
    n_progress = session.query(ReadingProgress).count()

print(f'Members         : {n_members:,}')
print(f'Messages        : {n_messages:,}')
print(f'BibleReferences : {n_refs:,}')
print(f'ReadingProgress : {n_progress:,}')

# Sanity assertions
assert n_members  > 0, 'No members inserted'
assert n_messages > 0, 'No messages inserted'
assert n_refs     > 0, 'No references inserted'
assert n_progress > 0, 'No reading progress inserted'

# Cross-check: progress rows >= refs (each ref expands to >= 1 chapter)
assert n_progress >= n_refs, f'Progress rows ({n_progress}) < refs ({n_refs}) — unexpected'

print('\n✅ All DB sanity checks passed.')

Members         : 47
Messages        : 257
BibleReferences : 272
ReadingProgress : 544

✅ All DB sanity checks passed.


### 3.2 Spot-check — Sample Inserted Rows

In [14]:
with get_session() as session:
    sample_progress = (
        session.query(ReadingProgress)
        .limit(10)
        .all()
    )
    for row in sample_progress:
        print(row)

<ReadingProgress member_id=1 Kejadian ch.1
<ReadingProgress member_id=1 Kejadian ch.2
<ReadingProgress member_id=2 Kejadian ch.1
<ReadingProgress member_id=2 Kejadian ch.2
<ReadingProgress member_id=3 Kejadian ch.1
<ReadingProgress member_id=3 Kejadian ch.2
<ReadingProgress member_id=4 Kejadian ch.1
<ReadingProgress member_id=4 Kejadian ch.2
<ReadingProgress member_id=5 Kejadian ch.1
<ReadingProgress member_id=5 Kejadian ch.2


---
## 4. Schedule

Validate the reading plan loaded correctly for the sampled dates.
Checks:
- Each sampled date has assigned chapters
- Cross-book days expand correctly
- Emoji loaded per date

In [ ]:
print('Reading plan validation\n')
for d in sorted(sample1_df['date'].unique()):
    assigned = bible_pipeline.schedule.get_by_date(d)
    emoji = bible_pipeline.schedule.get_emoji(d)
    if not assigned:
        print(f'  ⚠️  {d} — NO CHAPTERS ASSIGNED')
        continue
    books = sorted({book for book, _ in assigned})
    print(f'  {d}  {emoji}  {len(assigned):>3} chapters  books={books}')

Reading plan validation

  2020-08-03  💥    2 chapters  books=['Kejadian']
  2020-08-04  🐑    2 chapters  books=['Kejadian']
  2020-08-05  ⚡    2 chapters  books=['Kejadian']
  2020-08-06  🌊    2 chapters  books=['Kejadian']
  2020-08-07  🌈    2 chapters  books=['Kejadian']
  2020-08-08  🏯    2 chapters  books=['Kejadian']
  2020-08-09  🛡️    2 chapters  books=['Kejadian']


In [ ]:
# Cross-book day deep-check — print every chapter for days spanning >1 book
print('Cross-book day expansion:\n')
for d in sorted(sample1_df['date'].unique()):
    assigned = bible_pipeline.schedule.get_by_date(d)
    books = list(dict.fromkeys(book for book, _ in assigned))
    if len(books) > 1:
        print(f'{d}:')
        for book, ch in assigned:
            print(f'  {book} {ch}')
        print()

Cross-book day expansion:



---
## 5. Summarize

Print daily compliance for each sampled date.
Members who completed all assigned chapters for that day are marked with the day's emoji.

In [ ]:
for d in sorted(sample1_df['date'].unique()):
    print(f'\n── {d} ──')
    bible_pipeline.summarize(d)


── 2020-08-03 ──
📖 *Kej 1 - 2* 📖
1. Agnes 💥
2. Andrie HG 
3. BL katering 💥
4. Bento 💥
5. Ci Ina Paperku 
6. Darius Handoko 💥
7. Dewi Pratiwi 💥
8. Dicky Andrian 💥
9. Endang Surati 💥
10. Gracia 💥
11. Helen Fransiska Margarita 💥
12. Ibu Kartika 💥
13. Ivan teguh 💥
14. Ko Martin 
15. Kristin WIjaya Nusantara 💥
16. LF 
17. Lenny Pandjidharma 💥
18. Lidia 
19. Lindawati Haryanto 💥
20. Matthew Henry 2 💥
21. Melanie Chandra 💥
22. Mfitri 💥
23. Moses 💥
24. Nathanael Jeffry Teja 💥
25. Noah 💥
26. Nurcahaya Sihombing 
27. Oma Lisa 💥
28. Ruri Handoko 💥
29. Seto Ninik 💥
30. Sherly Cahyadi 💥
31. Sim Ay Tjan 💥
32. Tan 
33. Tejo Jayadi 💥
34. Tjunfebelyana 💥
35. Tom 💥
36. Villas Bakery 💥
37. Wesley and Mom😇🥰 💥
38. Wiwik 💥
39. Yoppie 💥
40. Yoshua 
41. Yozef Tjandra 💥
42. dr. Andreas C.N., Sp.B. 💥
43. no vi3ika 💥
44. piccer 💥
45. scarlet 💥
46. 🎍 💥
47. 🪸Martha 🍁 💥

── 2020-08-04 ──
📖 *Kej 3 - 4* 📖
1. Agnes 🐑
2. Andrie HG 🐑
3. BL katering 🐑
4. Bento 🐑
5. Ci Ina Paperku Kej 2
6. Darius Handoko 🐑
7. Dewi Pratiw